# Risk Assessor v5 — Standalone 노트북

DLthon Motorcycle Night Ride 위험도 평가 시스템 v5 의 self-contained 배포 버전.
**노트북 한 개만 있으면 동작** — 외부 .py 파일 import 불필요.

---

## 무엇을 하나

단일 프레임을 입력하면 **위험 등급 (안전/주의/위험/치명)** 과 **4개 경고 신호 (FCW/LDW/BSW/HEAD)** 를 출력합니다.

## 누가 쓰나

> **이 노트북의 주 사용자는 "다른 아키텍처로 직접 segmentation 을 학습한 팀원"** 입니다.
> 본인 모델의 7-class mask 를 주입하면 위험도 평가 파이프라인이 그 위에서 동작합니다.
> 우리 팀이 학습한 U-Net R34 (5-fold) 체크포인트를 쓰는 경로도 선택으로 제공합니다.

## 7-class mask 규약

본인 모델 출력이 아래 클래스 id 와 맞아야 합니다 (H×W, uint8):

| id | 이름 | 용도 |
|---|---|---|
| 0 | Background | — |
| 1 | Undrivable | — |
| 2 | Road | VP / corridor |
| 3 | Lane Mark | lane 선 / LDW |
| 4 | Moveable | 타 차량·보행자 (FCW/BSW/HEAD 대상) |
| 5 | My bike | ego anchor |
| 6 | Rider | ego anchor |

## 핵심 구성

1. **Segmentation** — 외부 mask 주입 (**기본, 권장**) / 내장 U-Net R34 5-fold (옵션)
2. **Depth** — Apple Depth Pro (metric m, 기본) / DA V2 (fallback) / 없음
3. **VP & Lane** — Method L (lane pairwise median)
4. **Distance** — 3-estimator fusion (ground + size + depth)
5. **Zone Corridor** — 4 nested zone (critical/danger/caution + sidelobe)
6. **4 Warning** — FCW (정면), LDW (이탈), BSW (측면), HEAD (역주행). 최종 bin = max(severities)

## 사용 한 줄 (기본: 외부 mask 주입)

```python
# 팀원 모델로 segment 한 mask (H, W) uint8, 위 7-class 규약
my_mask = my_segmentation_model(image)  # 본인 모델 호출

assessor = RiskAssessorV5(seg_checkpoints=None, use_depth='depth_pro')
result = assessor.assess_with_mask('frame.png', my_mask)
print(result['bin'], result['active_warnings'])
```

## 사용 한 줄 (옵션: 우리 팀 내장 U-Net)

```python
assessor = RiskAssessorV5(seg_checkpoints=OUR_CKPT_PATHS, use_depth='depth_pro')
result = assessor.assess('frame.png')
```


---

# 1. 환경 / 의존성

## 1.1 Python 버전

**Python 3.9 ~ 3.13 호환** (3.12 메인 타깃). Type hint 는 보수적으로 작성 — `from __future__ import annotations` 사용.

## 1.2 패키지 설치 (한 번만)

```bash
pip install torch torchvision \
    segmentation-models-pytorch \
    transformers>=4.40 \
    albumentations \
    opencv-python \
    pillow matplotlib numpy
```

**참고**:
- `transformers >= 4.40` 이 Apple Depth Pro 지원 시작 버전 (Python 3.12 환경이면 보통 자동으로 최신 설치됨)
- `torch` 는 CUDA 버전 맞춰서 [PyTorch 공식 install 명령](https://pytorch.org/get-started/locally/) 권장
- `segmentation-models-pytorch` 는 `smp` 로도 부르는 그 라이브러리

## 1.3 모델 가중치 다운로드

### Segmentation (수동)
최종 모델인 Lane Focus 5-fold 체크포인트 5개 (각 ~93MB) 가 필요합니다:
- `lane_focus_fold0_best.pth` ~ `lane_focus_fold4_best.pth`

이 파일들은 Hugging Face Hub(`minoak/motorcycle-night-lane-focus`)에서 받는다. 노트북과 같은 폴더 또는 임의 경로 가능.

### Depth (자동)
- **Apple Depth Pro** (`apple/DepthPro-hf`) — HuggingFace Hub 자동 다운로드 (~1GB, 캐시: `~/.cache/huggingface/hub/`)
- **DA V2** (`depth-anything/Depth-Anything-V2-Small-hf`) — 자동 (~100MB)

첫 실행 시 1~5분 소요, 이후 캐시 사용.

## 1.4 GPU / VRAM 요구

| 모드 | VRAM | 비고 |
|---|---|---|
| Apple Depth Pro (fp16) + 5-fold seg | ~5GB | 권장 |
| DA V2 + 5-fold seg | ~3GB | 가벼움 |
| Depth 없음 + 단일 fold | ~1.5GB | 최소 |

CPU 도 동작하지만 느림 (frame 당 ~5초 vs GPU 0.5초).

## 1.5 CONFIG

마지막 코드 셀의 사용 예시에서 본인 환경에 맞춰 다음 값만 수정하면 됩니다:
- `CKPT_PATHS` — 체크포인트 경로 리스트 (또는 단일 문자열)
- `DEPTH_MODE` — `'depth_pro'` / `'da_v2'` / `None`
- `IMAGE_PATH` — 평가할 이미지

---

# 2. Imports + 호환성 + 상수

Python 버전 체크, `torch.load` 호환 wrapper, 한글 폰트 자동 설정, 모든 상수 정의.

In [ ]:
from __future__ import annotations
import sys, os, json, time, warnings, inspect
from pathlib import Path
from itertools import combinations
from dataclasses import dataclass, field
from typing import Union, Dict, Optional, List, Tuple

# Python 버전 체크
if sys.version_info < (3, 9):
    raise RuntimeError(
        f'Python 3.9+ 필요 (현재 {sys.version_info.major}.{sys.version_info.minor}). '
        f'3.10 ~ 3.13 권장.')

import numpy as np
import cv2
import torch
import torch.nn as nn
from PIL import Image

warnings.filterwarnings('ignore')


def _safe_torch_load(path, map_location='cpu'):
    """PyTorch 버전별 torch.load 호환.
    - 2.4+: weights_only 명시 권장 (deprecation 경고 회피)
    - 2.6+: 기본값이 True 로 변경 → 반드시 False 지정
    - 2.3 이하: weights_only 파라미터 없음
    """
    sig = inspect.signature(torch.load)
    if 'weights_only' in sig.parameters:
        return torch.load(path, map_location=map_location, weights_only=False)
    return torch.load(path, map_location=map_location)


def _setup_korean_font():
    """한글 폰트 자동 설정 (Malgun Gothic → NanumGothic → AppleGothic → 기본 순)."""
    try:
        import matplotlib
        from matplotlib import font_manager
        available = {f.name for f in font_manager.fontManager.ttflist}
        for c in ['Malgun Gothic', 'NanumGothic', 'AppleGothic',
                  'Noto Sans CJK KR', 'DejaVu Sans']:
            if c in available:
                matplotlib.rcParams['font.family'] = c
                break
        matplotlib.rcParams['axes.unicode_minus'] = False
    except Exception:
        pass


_setup_korean_font()

# ====== Class & resolution ======
NUM_CLASSES = 7
IMG_SIZE = 768
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]
CLASS_NAMES = ['Background', 'Undrivable', 'Road', 'Lane Mark',
               'Moveable', 'My bike', 'Rider']

# ====== Distance priors ======
LANE_WIDTH_M = 3.5            # 한국 차선 표준
CAMERA_HEIGHT_M = 1.0         # 오토바이 dashcam 높이
DEFAULT_FOCAL_RATIO = 1.2     # f_default = img_w * 1.2 (광각 fallback)

OBJECT_WIDTH_M = {
    'moveable': 1.8, 'car': 1.8, 'truck': 2.5, 'bus': 2.55,
    'motorcycle': 0.8, 'bicycle': 0.6, 'person': 0.5,
}

DISTANCE_BINS = [(0, 5), (5, 15), (15, 30), (30, float('inf'))]
DISTANCE_BIN_NAMES = ['<5m', '5-15m', '15-30m', '>30m']

# Estimator 가중치 (depth 모드에 따라 다름)
ESTIMATOR_WEIGHTS_INVERSE = {'ground': 0.55, 'size': 0.40, 'depth': 0.05}  # DA V2
ESTIMATOR_WEIGHTS_METRIC = {'ground': 0.25, 'size': 0.25, 'depth': 0.50}   # Depth Pro

# ====== 4 Warning thresholds ======
FCW_DIST_CAUTION = 10.0
FCW_DIST_WARNING = 5.0
FCW_DIST_CRITICAL = 3.0
FCW_BBOX_RATIO_CRITICAL = 0.25

LDW_LATERAL_CAUTION = 0.15
LDW_LATERAL_WARNING = 0.3

BSW_DIST_CAUTION = 5.0
BSW_DIST_WARNING = 3.0

HEAD_DIST_WARNING = 5.0
HEAD_DIST_CRITICAL = 3.0
HEAD_ASPECT_MIN = 1.5
HEAD_BBOX_RATIO = 0.10

# ====== Zone alphas ======
DEFAULT_ALPHAS = {'critical': 0.85, 'danger': 0.55, 'caution': 0.15}
SIDE_EXPAND = 0.30

# ====== 색상 & 라벨 ======
WARNING_COLORS = {
    'FCW': '#FF4444', 'LDW': '#FFD700', 'BSW': '#FF8C00', 'HEAD': '#DA70D6'}
BIN_COLORS = {
    '안전': '#2ECC71', '주의': '#F39C12', '위험': '#E74C3C', '치명': '#8E44AD'}
SEVERITY_LABEL = {0: '', 1: '주의', 2: '위험', 3: '치명'}
SEVERITY_TO_BIN = {0: '안전', 1: '주의', 2: '위험', 3: '치명'}
DIR_KOR = {'L': '좌측', 'C': '정면', 'R': '우측'}

print(f'[v5] Python {sys.version_info.major}.{sys.version_info.minor}')
print(f'[v5] torch {torch.__version__}, CUDA available: {torch.cuda.is_available()}')

# ====== 최소 유효 거리 필터 (ego 오분류 방어) ======
# 2m 이내 거리는 바이크 본체 / 라이더 / 핸들바 / 사이드미러 반사 등으로 간주하여
# 모든 경고 대상에서 제외한다.
# 근거:
#   FCW/HEAD : 바이크 전후 길이 1.8~2m 안쪽은 내 바이크
#   BSW      : 차 폭 1.8m / 바이크 폭 0.8m — 옆 차량 반대편 엣지 기준
MIN_DIST_FCW = 2.0
MIN_DIST_BSW = 2.0
MIN_DIST_HEAD = 2.0


---

# 3. Lane Detection + VP (Method L)

Lane Mark (class 3) 픽셀에서 Hough → cluster → unified line. Pairwise 교점 length-weighted median 으로 VP. Ego lane 좌/우 선 식별 + symmetry fallback.

In [ ]:
def detect_lane_lines(mask: np.ndarray, min_seg: int = 30) -> list:
    """Lane mark (class 3) → Hough → cluster → unified lines.
    각 line dict: {dydx, c, y_top, y_bot, length}."""
    lane_bin = (mask == 3).astype(np.uint8) * 255
    if lane_bin.sum() < 100:
        return []

    k = np.ones((3, 3), np.uint8)
    lane_clean = cv2.morphologyEx(lane_bin, cv2.MORPH_OPEN, k)

    lines_raw = cv2.HoughLinesP(
        lane_clean, rho=1, theta=np.pi / 180,
        threshold=40, minLineLength=min_seg, maxLineGap=25)
    if lines_raw is None:
        return []

    segs = []
    for l in lines_raw:
        x1, y1, x2, y2 = l[0]
        if abs(x2 - x1) < 2 and abs(y2 - y1) < 2: continue
        if abs(y2 - y1) < 5: continue  # 수평선 제외
        dydx = (x2 - x1) / (y2 - y1)
        c = x1 - dydx * y1
        length = float(np.hypot(x2 - x1, y2 - y1))
        y_top, y_bot = min(y1, y2), max(y1, y2)
        x_mid = dydx * ((y_top + y_bot) / 2) + c
        segs.append((dydx, c, y_top, y_bot, length, x_mid))

    segs.sort(key=lambda s: -s[4])
    clusters = []
    for s in segs:
        dydx, c, y_top, y_bot, length, x_mid = s
        merged = False
        for cl in clusters:
            if abs(cl['dydx_avg'] - dydx) < 0.2 and abs(cl['x_mid_avg'] - x_mid) < 40:
                cl['members'].append(s)
                total_len = sum(m[4] for m in cl['members'])
                cl['dydx_avg'] = sum(m[0] * m[4] for m in cl['members']) / total_len
                cl['x_mid_avg'] = sum(m[5] * m[4] for m in cl['members']) / total_len
                merged = True
                break
        if not merged:
            clusters.append({'members': [s], 'dydx_avg': dydx, 'x_mid_avg': x_mid})

    lines = []
    for cl in clusters:
        members = cl['members']
        total_len = sum(m[4] for m in members)
        if total_len < 40: continue
        dydx = cl['dydx_avg']
        c_avg = sum(m[1] * m[4] for m in members) / total_len
        y_top = min(m[2] for m in members)
        y_bot = max(m[3] for m in members)
        lines.append({
            'dydx': float(dydx),
            'c': float(c_avg),
            'y_top': float(y_top),
            'y_bot': float(y_bot),
            'length': float(total_len),
        })
    return lines


def line_x_at(line: dict, y: float) -> float:
    return line['dydx'] * y + line['c']


def vp_from_lanes(lines: list, W: int, H: int) -> tuple:
    """Method L — pairwise 교점 length-weighted median.
    Returns (vp_x, vp_y, confidence)."""
    if len(lines) < 2:
        return W // 2, int(H * 0.4), 0.0

    inters = []
    weights = []
    for i, j in combinations(range(len(lines)), 2):
        l1, l2 = lines[i], lines[j]
        if abs(l2['dydx'] - l1['dydx']) < 1e-4: continue
        vy = (l1['c'] - l2['c']) / (l2['dydx'] - l1['dydx'])
        vx = l1['dydx'] * vy + l1['c']
        if not (-W * 0.2 <= vx <= W * 1.2): continue
        if not (-H * 0.5 <= vy <= H * 0.8): continue
        inters.append((vx, vy))
        weights.append(l1['length'] * l2['length'])

    if len(inters) == 0:
        return W // 2, int(H * 0.4), 0.0

    inters = np.array(inters)
    weights = np.array(weights)
    weights /= weights.sum()
    order_x = np.argsort(inters[:, 0])
    cw = np.cumsum(weights[order_x])
    vx = inters[order_x[np.searchsorted(cw, 0.5)], 0]
    order_y = np.argsort(inters[:, 1])
    cw = np.cumsum(weights[order_y])
    vy = inters[order_y[np.searchsorted(cw, 0.5)], 1]

    std_x = float(np.std(inters[:, 0])) / W
    conf = float(np.clip(1.0 - std_x * 2, 0.0, 1.0))
    return int(vx), int(vy), conf


def identify_ego_lane(lines: list, W: int, H: int, sample_y: float = None) -> dict:
    """이미지 하단에서 W/2 주변 좌/우 가장 가까운 line 한 쌍.
    Returns {left, right, conf}. Symmetry fallback 포함."""
    if len(lines) == 0:
        return {'left': None, 'right': None, 'conf': 0.0}

    sy = sample_y if sample_y is not None else H * 0.85
    left_cands, right_cands = [], []
    for l in lines:
        x_at_sy = l['dydx'] * sy + l['c']
        if x_at_sy < W * 0.5:
            left_cands.append((abs(x_at_sy - W * 0.5), l))
        else:
            right_cands.append((abs(x_at_sy - W * 0.5), l))

    left = min(left_cands, key=lambda t: t[0])[1] if left_cands else None
    right = min(right_cands, key=lambda t: t[0])[1] if right_cands else None

    if left and right:
        return {'left': left, 'right': right, 'conf': 1.0}
    elif left:
        right_synth = {
            'dydx': -left['dydx'],
            'c': W - left['c'],
            'y_top': left['y_top'], 'y_bot': left['y_bot'],
            'length': left['length'] * 0.5,
        }
        return {'left': left, 'right': right_synth, 'conf': 0.5}
    elif right:
        left_synth = {
            'dydx': -right['dydx'],
            'c': W - right['c'],
            'y_top': right['y_top'], 'y_bot': right['y_bot'],
            'length': right['length'] * 0.5,
        }
        return {'left': left_synth, 'right': right, 'conf': 0.5}
    return {'left': None, 'right': None, 'conf': 0.0}


def check_lane_viol_strict(bbox, ego: dict, H: int) -> str:
    """bbox 가 ego lane 의 좌/우 line 가로지름 → 'from_left'/'from_right'/''."""
    if ego['left'] is None or ego['right'] is None:
        return ''
    x0, y0, x1, y1 = bbox
    sample_ys = np.linspace(y0, y1, 10)
    straddle_left = 0
    straddle_right = 0
    for y in sample_ys:
        lx = line_x_at(ego['left'], y)
        rx = line_x_at(ego['right'], y)
        if x0 < lx < x1: straddle_left += 1
        if x0 < rx < x1: straddle_right += 1
    if straddle_left >= 3: return 'from_left'
    if straddle_right >= 3: return 'from_right'
    return ''


def check_lane_viol_x1(bbox, W: int) -> str:
    """이미지 중앙 가로지름 + 너비 > 30%W → 'from_left'/'from_right'/''."""
    x0, y0, x1, y1 = bbox
    bw = x1 - x0
    if bw < 0.30 * W: return ''
    if not (x0 < W / 2 < x1): return ''
    cx = (x0 + x1) / 2
    return 'from_right' if cx > W / 2 else 'from_left'

---

# 4. Object Extraction

Mask 의 Moveable / My bike / Rider (class 4/5/6) 를 connected components 로 분리. min_area 200 픽셀 필터.

In [ ]:
def extract_objects(mask: np.ndarray, min_area: int = 200) -> list:
    """Connected Components 로 객체 분리.
    Returns: list of {class_id, class_name, bbox, area, cx, cy}."""
    objs = []
    for cid in (4, 5, 6):
        bin_m = (mask == cid).astype(np.uint8)
        if bin_m.sum() < min_area: continue
        n, lbl = cv2.connectedComponents(bin_m, connectivity=8)
        for k in range(1, n):
            ys, xs = np.where(lbl == k)
            if len(xs) < min_area: continue
            x0, x1 = int(xs.min()), int(xs.max())
            y0, y1 = int(ys.min()), int(ys.max())
            objs.append({
                'class_id': int(cid),
                'class_name': CLASS_NAMES[cid],
                'bbox': (x0, y0, x1, y1),
                'area': int(len(xs)),
                'cx': int((x0 + x1) / 2),
                'cy': int((y0 + y1) / 2),
            })
    return objs

---

# 5. Distance Estimator (3-Estimator Fusion)

**3 estimators**: ground (`f·h/(y_b-vp_y)`), size (`f·W_real/w_px`), depth (median).  
**Outlier reject** (3× median 밖 제거) + **weighted mean**. Apple Depth Pro metric 모드와 DA V2 inverse 모드 가중치 분리.  
Focal length 는 (1) Depth Pro 자동 추정 우선, (2) 차선폭 prior 역산, (3) 이미지 너비 fallback 순.

In [ ]:
def estimate_focal_from_lanes(lanes, vp_y, img_w, img_h, min_sample=2):
    """차선폭 3.5m prior 로 focal 역산. Mobileye Gen 1 방식."""
    if len(lanes) < 2: return None

    cx = img_w / 2
    # VP 근사로 중앙 거리 체크
    if len(lanes) >= 2:
        xs = [l['dydx'] * vp_y + l['c'] for l in lanes]
        approx_vp_x = float(np.median(xs))
        if abs(approx_vp_x - cx) > img_w * 0.2:
            return None  # VP 가 너무 치우치면 공식 성립 안 함

    y_sample = img_h * 0.9
    left_lane, right_lane = None, None
    for l in lanes:
        x_sample = l['dydx'] * y_sample + l['c']
        if x_sample < cx - 20:
            if left_lane is None or abs(x_sample - cx) < abs(left_lane['dydx'] * y_sample + left_lane['c'] - cx):
                left_lane = l
        elif x_sample > cx + 20:
            if right_lane is None or abs(x_sample - cx) < abs(right_lane['dydx'] * y_sample + right_lane['c'] - cx):
                right_lane = l
    if left_lane is None or right_lane is None: return None

    y_check = img_h * 0.9
    xl_check = left_lane['dydx'] * y_check + left_lane['c']
    xr_check = right_lane['dydx'] * y_check + right_lane['c']
    if xl_check >= xr_check: return None

    consts = []
    for y_frac in (0.75, 0.85, 0.95):
        y = img_h * y_frac
        if y <= vp_y + 20: continue
        x_left = left_lane['dydx'] * y + left_lane['c']
        x_right = right_lane['dydx'] * y + right_lane['c']
        w_px = x_right - x_left
        if w_px < 20: continue
        if x_left < -img_w * 0.5 or x_right > img_w * 1.5: continue
        const = w_px * (y - vp_y)
        if const <= 0: continue
        consts.append(const)
    if len(consts) < min_sample: return None

    const = float(np.median(consts))
    f_px = const * CAMERA_HEIGHT_M / LANE_WIDTH_M
    if not (img_w * 0.3 < f_px < img_w * 2.5): return None
    return float(f_px)


def distance_from_ground(bbox, vp_y, f_px, camera_h=CAMERA_HEIGHT_M):
    """d = f_px · h / (y_bottom - vp_y)."""
    x0, y0, x1, y1 = bbox
    dy = y1 - vp_y
    if dy <= 5: return None
    d = f_px * camera_h / dy
    if not (0.5 < d < 200): return None
    return float(d)


def classify_object(bbox, class_id):
    """bbox aspect → coarse class ('car', 'truck', 'motorcycle', etc)."""
    x0, y0, x1, y1 = bbox
    w = max(x1 - x0, 1); h = max(y1 - y0, 1)
    ar = w / h
    if class_id == 5: return 'motorcycle'
    if class_id == 6: return 'person'
    if ar > 2.5: return 'truck'
    if ar > 0.7: return 'car'
    return 'motorcycle'


def distance_from_size(bbox, obj_class, f_px):
    """d = f_px · W_real / w_bbox_px."""
    x0, y0, x1, y1 = bbox
    w_px = max(x1 - x0, 1)
    W_real = OBJECT_WIDTH_M.get(obj_class, 1.8)
    d = f_px * W_real / w_px
    if not (0.5 < d < 200): return None
    return float(d)


def distance_from_depth(depth_map, bbox, alpha=30.0, beta=0.0, pct=90, is_metric=False):
    """is_metric=True (Depth Pro): bbox median (m).
    is_metric=False (DA V2): inverse depth → α/p + β."""
    x0, y0, x1, y1 = bbox
    patch = depth_map[y0:y1+1, x0:x1+1]
    if patch.size == 0: return None

    if is_metric:
        valid = patch[(patch > 0.3) & (patch < 100)]
        if valid.size < 5: return None
        d = float(np.median(valid))
        if not (0.3 < d < 100): return None
        return d

    p = float(np.percentile(patch, pct)) / 255.0
    if p < 0.05: return None
    d = alpha / p + beta
    if not (0.5 < d < 200): return None
    return float(d)


def fuse_distances(estimates, weights=None, depth_is_metric=False):
    """3-estimator fusion with outlier reject + weighted mean."""
    if weights is None:
        weights = ESTIMATOR_WEIGHTS_METRIC if depth_is_metric else ESTIMATOR_WEIGHTS_INVERSE

    valid = {k: v for k, v in estimates.items() if v is not None}
    n_valid = len(valid)
    if n_valid == 0:
        return {'d_fused_m': None, 'd_bin': 'unknown', 'd_bin_idx': -1,
                'confidence': 'none', 'n_valid': 0, 'estimates': estimates}

    values = list(valid.values())
    med = float(np.median(values))
    filtered = {k: v for k, v in valid.items() if 0.33 * med < v < 3.0 * med}
    if len(filtered) == 0: filtered = valid

    total_w = sum(weights[k] for k in filtered)
    d_fused = sum(filtered[k] * weights[k] / total_w for k in filtered)

    d_bin_idx = 3
    for i, (lo, hi) in enumerate(DISTANCE_BINS):
        if lo <= d_fused < hi:
            d_bin_idx = i; break

    bins_agreed = set()
    for v in valid.values():
        for i, (lo, hi) in enumerate(DISTANCE_BINS):
            if lo <= v < hi: bins_agreed.add(i); break

    if n_valid >= 3 and len(bins_agreed) == 1: conf = 'high'
    elif n_valid >= 2 and d_bin_idx in bins_agreed: conf = 'medium'
    elif n_valid >= 1: conf = 'low'
    else: conf = 'none'

    return {
        'd_fused_m': round(d_fused, 2),
        'd_bin': DISTANCE_BIN_NAMES[d_bin_idx],
        'd_bin_idx': d_bin_idx,
        'confidence': conf,
        'n_valid': n_valid,
        'estimates': {k: round(v, 2) if v else None for k, v in estimates.items()},
    }


class DistanceEstimator:
    """3-estimator fusion 거리 추정기."""

    def __init__(self, depth_is_metric: bool = False):
        self.f_px: Optional[float] = None
        self.camera_h: float = CAMERA_HEIGHT_M
        self.alpha: float = 30.0
        self.beta: float = 0.0
        self.depth_is_metric: bool = depth_is_metric

    def calibrate(self, lanes, vp_y, img_w, img_h):
        f = estimate_focal_from_lanes(lanes, vp_y, img_w, img_h)
        if f is None:
            self.f_px = img_w * DEFAULT_FOCAL_RATIO
            return False
        self.f_px = f
        return True

    def calibrate_depth_alpha(self, depth_map, moveables, vp_y):
        """DA V2 α,β 를 d_ground 로 fit. Depth Pro 면 skip."""
        if self.depth_is_metric: return
        if self.f_px is None or len(moveables) < 2: return
        pts = []
        for o in moveables:
            d_g = distance_from_ground(o['bbox'], vp_y, self.f_px, self.camera_h)
            if d_g is None: continue
            x0, y0, x1, y1 = o['bbox']
            patch = depth_map[y0:y1+1, x0:x1+1]
            if patch.size == 0: continue
            p = float(np.percentile(patch, 90)) / 255.0
            if p < 0.05: continue
            pts.append((1.0 / p, d_g))
        if len(pts) < 2: return
        X = np.array([p[0] for p in pts])
        Y = np.array([p[1] for p in pts])
        try:
            alpha, beta = np.polyfit(X, Y, 1)
            if 1 < alpha < 200:
                self.alpha = float(alpha)
                self.beta = float(beta)
        except Exception:
            pass

    def estimate(self, bbox, class_id, depth_map, vp_y):
        if self.f_px is None:
            self.f_px = float(DEFAULT_FOCAL_RATIO * 1000)  # 대략 fallback

        obj_class = classify_object(bbox, class_id)
        d_ground = distance_from_ground(bbox, vp_y, self.f_px, self.camera_h)
        d_size = distance_from_size(bbox, obj_class, self.f_px)
        d_depth = None
        if depth_map is not None:
            d_depth = distance_from_depth(
                depth_map, bbox, self.alpha, self.beta,
                is_metric=self.depth_is_metric)
        fused = fuse_distances({'ground': d_ground, 'size': d_size, 'depth': d_depth},
                               depth_is_metric=self.depth_is_metric)
        fused['obj_class'] = obj_class
        return fused

---

# 6. 4-Zone Ego Corridor

VP + ego_lane 하단 (Lx, Rx) 으로 4 nested 사다리꼴.  
**Anchor** = bbox bottom-center, **Soft promote** 30% (더 안쪽 zone 에 30%+ 겹치면 승급).  
**Sidelobe** 별도 신호 (BSW 용, ratio > 0.40 hit).

In [ ]:
def _y_at(vp_y, alpha, H):
    return vp_y + alpha * (H - 1 - vp_y)


def _lane_x_at_zone(vp_x, vp_y, x_bottom, H, y):
    denom = max(H - 1 - vp_y, 1e-6)
    t = (y - vp_y) / denom
    return vp_x + t * (x_bottom - vp_x)


def build_corridor(vp_x, vp_y, x_left_bottom, x_right_bottom, H, W,
                   vp_conf=1.0, alphas=None, side_expand=SIDE_EXPAND):
    """4-zone nested polygon + sidelobe.
    Returns dict 또는 None (vp_conf < 0.3 또는 lane_w < 20 시)."""
    if vp_conf < 0.3: return None
    if alphas is None: alphas = DEFAULT_ALPHAS

    y_bot = H - 1
    Lx, Rx = x_left_bottom, x_right_bottom
    if Rx - Lx < 20: return None

    def trapezoid(alpha_top):
        yt = _y_at(vp_y, alpha_top, H)
        xtl = _lane_x_at_zone(vp_x, vp_y, Lx, H, yt)
        xtr = _lane_x_at_zone(vp_x, vp_y, Rx, H, yt)
        return np.array([[xtl, yt], [xtr, yt],
                         [Rx, y_bot], [Lx, y_bot]], dtype=np.int32)

    polys = {
        'critical': trapezoid(alphas['critical']),
        'danger':   trapezoid(alphas['danger']),
        'caution':  trapezoid(alphas['caution']),
    }
    lane_w_bot = Rx - Lx
    dx = side_expand * lane_w_bot
    polys['sidelobe_left'] = np.array(
        [[vp_x, vp_y], [Lx, y_bot], [Lx - dx, y_bot]], dtype=np.int32)
    polys['sidelobe_right'] = np.array(
        [[vp_x, vp_y], [Rx + dx, y_bot], [Rx, y_bot]], dtype=np.int32)

    masks = {}
    for name, poly in polys.items():
        m = np.zeros((H, W), dtype=np.uint8)
        cv2.fillPoly(m, [poly], 1)
        masks[name] = m

    return {'polygons': polys, 'masks': masks,
            'vp': (int(vp_x), int(vp_y)),
            'lane_bottom': (int(Lx), int(Rx)),
            'alphas': alphas}


def classify_zone(bbox, corridor, soft_promote_ratio=0.30):
    """Bbox → severity (0=safe, 1=caution, 2=danger, 3=critical)."""
    info = {
        'anchor_zone': 'safe', 'bbox_ratios': {},
        'sidelobe_hit': None, 'sidelobe_ratio': 0.0, 'in_corridor': False,
    }
    if corridor is None: return 0, info

    x0, y0, x1, y1 = bbox
    H, W = corridor['masks']['critical'].shape
    foot_x = max(0, min(W - 1, int((x0 + x1) / 2)))
    foot_y = max(0, min(H - 1, int(y1)))

    anchor = 'safe'
    for name in ('critical', 'danger', 'caution'):
        if corridor['masks'][name][foot_y, foot_x] > 0:
            anchor = name; break
    info['anchor_zone'] = anchor
    info['in_corridor'] = anchor != 'safe'

    bbox_area = max((x1 - x0) * (y1 - y0), 1)
    cx0, cy0 = max(0, x0), max(0, y0)
    cx1, cy1 = min(W, x1), min(H, y1)
    if cx1 <= cx0 or cy1 <= cy0: return 0, info

    for name in ('critical', 'danger', 'caution',
                 'sidelobe_left', 'sidelobe_right'):
        patch = corridor['masks'][name][cy0:cy1, cx0:cx1]
        info['bbox_ratios'][name] = round(int(patch.sum()) / bbox_area, 3)

    sev_map = {'safe': 0, 'caution': 1, 'danger': 2, 'critical': 3}
    sev = sev_map[anchor]

    if info['bbox_ratios']['critical'] > soft_promote_ratio:
        sev = max(sev, 3)
    elif info['bbox_ratios']['danger'] > soft_promote_ratio:
        sev = max(sev, 2)
    elif info['bbox_ratios']['caution'] > soft_promote_ratio:
        sev = max(sev, 1)

    sl_max = max(info['bbox_ratios']['sidelobe_left'],
                 info['bbox_ratios']['sidelobe_right'])
    if sl_max > 0.40:
        info['sidelobe_hit'] = ('left' if info['bbox_ratios']['sidelobe_left']
                                > info['bbox_ratios']['sidelobe_right'] else 'right')
        info['sidelobe_ratio'] = round(sl_max, 3)

    return sev, info


def draw_zones_on_ax(ax, corridor, alpha=0.15):
    if corridor is None: return
    import matplotlib.patches as mpatches
    zone_colors = {
        'critical':       ('red',    0.20),
        'danger':         ('orange', 0.15),
        'caution':        ('yellow', 0.08),
        'sidelobe_left':  ('blue',   0.08),
        'sidelobe_right': ('blue',   0.08),
    }
    draw_order = ['caution', 'danger', 'critical',
                  'sidelobe_left', 'sidelobe_right']
    for name in draw_order:
        if name not in corridor['polygons']: continue
        poly = corridor['polygons'][name]
        color, a = zone_colors[name]
        ax.add_patch(mpatches.Polygon(poly, closed=True,
                                       facecolor=color, alpha=a,
                                       edgecolor=color, linewidth=0.8))

---

# 7. 4 Warning Modules

**FCW** (정면 충돌), **LDW** (이탈), **BSW** (측면), **HEAD** (역주행). 각 Level 0~3.  
최종 bin = `max(FCW.level, LDW.level, BSW.level, HEAD.level)`.  
Cut-in 과 Head-on 분리: HEAD 는 `0.4 ≤ cx/W ≤ 0.6` AND `aspect ≥ 1.8` 일 때만 발화 (cut-in 은 BSW 로).

In [ ]:
@dataclass
class WarningResult:
    name: str
    level: int
    reason: str = ''
    bbox: Optional[tuple] = None
    extra: dict = field(default_factory=dict)

    def to_dict(self):
        return {'name': self.name, 'level': self.level, 'reason': self.reason,
                'bbox': list(self.bbox) if self.bbox else None, **self.extra}


@dataclass
class V5State:
    fcw: WarningResult
    ldw: WarningResult
    bsw: WarningResult
    head: WarningResult
    d_front_m: Optional[float] = None
    d_left_m: Optional[float] = None
    d_right_m: Optional[float] = None
    f_calibrated: bool = False
    f_px: Optional[float] = None

    @property
    def max_severity(self) -> int:
        return max(self.fcw.level, self.ldw.level, self.bsw.level, self.head.level)

    @property
    def bin(self) -> str:
        return SEVERITY_TO_BIN[self.max_severity]

    @property
    def active_warnings(self) -> list:
        return [w for w in (self.fcw, self.ldw, self.bsw, self.head) if w.level > 0]

    @property
    def top_warning(self) -> Optional[WarningResult]:
        act = self.active_warnings
        if not act: return None
        return max(act, key=lambda w: w.level)


def detect_fcw(front_objs, img_w, img_h):
    """정면 가장 가까운 객체 기반."""
    if not front_objs: return WarningResult('FCW', 0)
    obj = min(front_objs, key=lambda o: o['dist_info']['d_fused_m'] or 999)
    d = obj['dist_info']['d_fused_m']
    if d is None: return WarningResult('FCW', 0, 'no distance estimate')

    x0, y0, x1, y1 = obj['bbox']
    area_ratio = ((x1 - x0) * (y1 - y0)) / (img_w * img_h)

    if d < FCW_DIST_CRITICAL or area_ratio > FCW_BBOX_RATIO_CRITICAL:
        return WarningResult('FCW', 3,
            f'정면 {d:.1f}m (bbox {area_ratio*100:.0f}%)',
            bbox=obj['bbox'],
            extra={'distance_m': round(d, 2), 'area_ratio': round(area_ratio, 3)})
    elif d < FCW_DIST_WARNING:
        return WarningResult('FCW', 2, f'정면 {d:.1f}m', bbox=obj['bbox'],
            extra={'distance_m': round(d, 2)})
    elif d < FCW_DIST_CAUTION:
        return WarningResult('FCW', 1, f'정면 {d:.1f}m', bbox=obj['bbox'],
            extra={'distance_m': round(d, 2)})
    return WarningResult('FCW', 0)


def detect_ldw(ego, img_w, img_h, vp_y, ego_x=None):
    """ego_x vs ego_lane center / lane_w 기반 편향·이탈."""
    if ego['left'] is None or ego['right'] is None:
        return WarningResult('LDW', 0, 'ego_lane 식별 실패')

    y_check = img_h * 0.95
    lx = line_x_at(ego['left'], y_check)
    rx = line_x_at(ego['right'], y_check)
    lane_w = rx - lx
    if lane_w <= 20:
        return WarningResult('LDW', 0, 'ego_lane 너비 이상')
    lane_center = (lx + rx) / 2
    if ego_x is None: ego_x = img_w / 2

    lateral_offset = ego_x - lane_center
    lateral_ratio = abs(lateral_offset) / lane_w

    # 측정 불가 guard (sus seg 방어)
    if lateral_ratio > 0.5:
        return WarningResult('LDW', 0,
            f'측정 불가 (편향 {lateral_ratio*100:.0f}%, 감지 오류 의심)',
            extra={'lateral_ratio': round(lateral_ratio, 3), 'unreliable': True})

    direction = '우측' if lateral_offset > 0 else '좌측'
    # L2 이탈 제거: 30% 이상 편향은 confidence 부족으로 측정 불가 처리
    # (single-frame 으로 이탈 확정 어려움, FP 방지)
    if lateral_ratio > LDW_LATERAL_WARNING or ego_x < lx or ego_x > rx:
        return WarningResult('LDW', 0,
            f'[측정 불가] {direction} 이탈 의심 ({lateral_ratio*100:.0f}%, 확정 불가)',
            extra={'lateral_ratio': round(lateral_ratio, 3),
                   'unreliable': True, 'reason_detail': 'possible_departure'})
    elif lateral_ratio > LDW_LATERAL_CAUTION:
        return WarningResult('LDW', 1, f'{direction} 편향 ({lateral_ratio*100:.0f}%)',
            extra={'lateral_ratio': round(lateral_ratio, 3),
                   'ego_x': int(ego_x), 'lane_center': int(lane_center)})
    return WarningResult('LDW', 0,
        extra={'lateral_ratio': round(lateral_ratio, 3)})


def detect_bsw(left_objs, right_objs, strict_violators, x1_violators):
    """측면 가장 가까운 객체 + 침범 여부."""
    side_objs = left_objs + right_objs
    if not side_objs: return WarningResult('BSW', 0)
    obj = min(side_objs, key=lambda o: o['dist_info']['d_fused_m'] or 999)
    d = obj['dist_info']['d_fused_m']
    if d is None: return WarningResult('BSW', 0, 'no distance')
    side = '좌측' if obj['cx_rel'] < 0.5 else '우측'
    violating = any(o['bbox'] == obj['bbox'] for o in strict_violators + x1_violators)

    if d < BSW_DIST_WARNING and violating:
        return WarningResult('BSW', 3, f'{side} 침범 {d:.1f}m', bbox=obj['bbox'],
            extra={'distance_m': round(d, 2), 'side': side, 'violating': True})
    elif d < BSW_DIST_WARNING:
        return WarningResult('BSW', 2, f'{side} 근접 {d:.1f}m', bbox=obj['bbox'],
            extra={'distance_m': round(d, 2), 'side': side})
    elif d < BSW_DIST_CAUTION:
        return WarningResult('BSW', 1, f'{side} {d:.1f}m', bbox=obj['bbox'],
            extra={'distance_m': round(d, 2), 'side': side})
    return WarningResult('BSW', 0)


def detect_head(front_objs, strict_violators, x1_violators, img_w, img_h):
    """정면 역주행 (cut-in 은 제외)."""
    if not front_objs: return WarningResult('HEAD', 0)
    violator_bboxes = [o['bbox'] for o in strict_violators + x1_violators]
    head_cands = [o for o in front_objs if o['bbox'] in violator_bboxes]
    if not head_cands: return WarningResult('HEAD', 0, 'no front violator')

    obj = min(head_cands, key=lambda o: o['dist_info']['d_fused_m'] or 999)
    d = obj['dist_info']['d_fused_m']
    if d is None: return WarningResult('HEAD', 0)

    x0, y0, x1, y1 = obj['bbox']
    bw, bh = max(x1 - x0, 1), max(y1 - y0, 1)
    aspect = bw / bh
    area_ratio = (bw * bh) / (img_w * img_h)
    cx_rel = ((x0 + x1) / 2) / img_w

    is_frontal_center = (0.45 <= cx_rel <= 0.55)   # 중앙 10% 만
    is_frontal_aspect = (aspect >= 2.0)           # 넓은 정면 차량만
    if not (is_frontal_center and is_frontal_aspect):
        return WarningResult('HEAD', 0,
            f'cut-in 가능성 (cx={cx_rel:.2f}, asp={aspect:.1f})',
            extra={'skipped_cut_in': True, 'cx_rel': round(cx_rel, 2),
                   'aspect': round(aspect, 2)})

    # 거리가 너무 멀면 HEAD 발동 안 함 (극보수화)
    if d is None or d > 4.0:
        return WarningResult('HEAD', 0, f'cut-in 가능성 (거리 {d}m)')

    if d < HEAD_DIST_CRITICAL:
        return WarningResult('HEAD', 3,
            f'역주행 정면 {d:.1f}m (aspect {aspect:.1f})', bbox=obj['bbox'],
            extra={'distance_m': round(d, 2), 'aspect': round(aspect, 2),
                   'area_ratio': round(area_ratio, 3)})
    elif d < HEAD_DIST_WARNING:
        return WarningResult('HEAD', 2, f'역주행 의심 {d:.1f}m', bbox=obj['bbox'],
            extra={'distance_m': round(d, 2)})
    return WarningResult('HEAD', 0)


def _warning_distance_m(w):
    """WarningResult 에서 거리(m) 추출 — extra 우선, 없으면 reason 파싱."""
    import re as _re
    extra = getattr(w, 'extra', None) or {}
    for k in ('distance_m', 'dist_m', 'd_m'):
        v = extra.get(k)
        if v is not None:
            try:
                return float(v)
            except (TypeError, ValueError):
                pass
    reason = getattr(w, 'reason', '') or ''
    m = _re.search(r'(\d+\.?\d*)\s*m', reason)
    return float(m.group(1)) if m else None


def suppress_short_range_warnings(state, min_valid=2.0):
    """2m 이내 거리는 ego 오분류로 간주:
      - 해당 warning 의 level 을 0 으로 downgrade
      - state.d_front_m / d_left_m / d_right_m 도 2m 미만이면 None 처리
    state 는 in-place 수정."""
    thresholds = {'FCW': MIN_DIST_FCW, 'BSW': MIN_DIST_BSW, 'HEAD': MIN_DIST_HEAD}
    for w in [state.fcw, state.bsw, state.head]:
        if w.level == 0:
            continue
        min_d = thresholds.get(w.name, min_valid)
        dist = _warning_distance_m(w)
        if dist is not None and dist < min_d:
            orig = w.reason or f'L{w.level}'
            w.level = 0
            w.reason = f'[억제] {orig} < {min_d}m (ego 오분류 의심)'
            w.bbox = None
            if w.extra is None:
                w.extra = {}
            w.extra['suppressed_short_range'] = True

    # 위치별 거리도 일관성 맞춤 — 2m 미만은 None 처리
    for attr in ('d_front_m', 'd_left_m', 'd_right_m'):
        v = getattr(state, attr, None)
        if v is not None and v < min_valid:
            try:
                setattr(state, attr, None)
            except (AttributeError, TypeError):
                pass


def promote_lane_violators_to_bsw(bsw, n_strict, n_x1):
    """차선 침범 (strict/x1 violator) 을 BSW 경고로 승격.
    detect_bsw 는 side_objs (좌/우 측면 객체) 기반이라 정면 등 다른 영역의
    lane 침범을 놓치는 경우가 있어 후처리로 보완.

    레벨 기준:
      - x1 violator (내 lane 깊숙히 침범) 1대 이상 → BSW 2 (위험)
      - strict violator (lane 경계 밟음) 1대 이상  → BSW 1 (주의)
    기존 bsw.level 이 더 높으면 유지.
    """
    # 완화된 기준: lane violator 만으로는 주의(L1) 까지만 승격.
    # 위험/치명 판정은 거리 기반 (detect_bsw) 에 맡긴다.
    if n_x1 >= 1 or n_strict >= 1:
        target = 1
    else:
        target = 0

    if target == 0 or bsw.level >= target:
        return

    reason_parts = []
    if n_x1: reason_parts.append(f'x1 {n_x1}대')
    if n_strict: reason_parts.append(f'strict {n_strict}대')
    bsw.level = target
    bsw.reason = f'차선 침범 — {", ".join(reason_parts)}'
    if bsw.extra is None:
        bsw.extra = {}
    bsw.extra['promoted_from_lane_viol'] = True
    bsw.extra['n_strict'] = n_strict
    bsw.extra['n_x1'] = n_x1


---

# 8. Segmentation + Depth Loaders

**Segmentation**: U-Net R34, 단일 또는 5-fold ensemble (majority vote per pixel).  
**Depth (3-mode)**:
- `'depth_pro'` — Apple Depth Pro (metric m + focal 자동, 1GB, fp16 권장)
- `'da_v2'` — Depth Anything V2 Small (inverse, ~100MB)
- `None` — Depth 없이 (거리 = ground + size only)

In [ ]:
def load_seg_model(ckpt_path, device='cuda'):
    """단일 fold 로드."""
    import segmentation_models_pytorch as smp
    model = smp.Unet('resnet34', encoder_weights=None,
                     in_channels=3, classes=NUM_CLASSES)
    model = model.to(device)
    ckpt = _safe_torch_load(ckpt_path, map_location=device)
    state = ckpt.get('state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state)
    model.eval()
    info = ({k: ckpt.get(k) for k in ('epoch', 'miou', 'lane_iou', 'fold')}
            if isinstance(ckpt, dict) else {})
    print(f'[v5] Seg 로드: {Path(ckpt_path).name} '
          f'(fold={info.get("fold")}, mIoU={info.get("miou")}, '
          f'lane_iou={info.get("lane_iou")})')
    return model


def load_seg_ensemble(ckpt_paths, device='cuda'):
    """여러 fold 로드 (없는 파일은 skip)."""
    models = []
    for p in ckpt_paths:
        if not Path(p).exists():
            print(f'[v5] 경고: {p} 없음 — skip')
            continue
        models.append(load_seg_model(str(p), device))
    print(f'[v5] Ensemble 로드 완료: {len(models)} folds')
    return models


def predict_seg(model, img_np, device='cuda'):
    """단일 모델 inference (768 resize → argmax → 원본 크기)."""
    import albumentations as A
    from albumentations.pytorch import ToTensorV2
    H, W = img_np.shape[:2]
    tfm = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])
    t = tfm(image=img_np)['image'].unsqueeze(0).to(device)
    with torch.no_grad():
        pred = model(t).argmax(1).squeeze(0).cpu().numpy().astype(np.uint8)
    return cv2.resize(pred, (W, H), interpolation=cv2.INTER_NEAREST)


def predict_seg_ensemble(models, img_np, device='cuda'):
    """5-fold majority vote per pixel."""
    if len(models) == 0:
        raise ValueError('No models in ensemble')
    if len(models) == 1:
        return predict_seg(models[0], img_np, device)
    H, W = img_np.shape[:2]
    votes = np.zeros((NUM_CLASSES, H, W), dtype=np.int32)
    for model in models:
        mask = predict_seg(model, img_np, device)
        for c in range(NUM_CLASSES):
            votes[c] += (mask == c).astype(np.int32)
    return np.argmax(votes, axis=0).astype(np.uint8)


def load_depth_pro(device='cuda'):
    """Apple Depth Pro — metric depth + focal 자동 추정."""
    from transformers import AutoModelForDepthEstimation, AutoImageProcessor
    proc = AutoImageProcessor.from_pretrained('apple/DepthPro-hf')
    dtype = torch.float16 if device == 'cuda' else torch.float32
    model = AutoModelForDepthEstimation.from_pretrained(
        'apple/DepthPro-hf', torch_dtype=dtype).to(device).eval()
    print('[v5] Apple Depth Pro 로드 (metric + focal 자동)')
    return {'kind': 'depth_pro', 'processor': proc, 'model': model, 'device': device}


def load_depth_v2(device='cuda'):
    """Depth Anything V2 Small (legacy/fallback)."""
    from transformers import pipeline
    pipe = pipeline('depth-estimation',
                    model='depth-anything/Depth-Anything-V2-Small-hf',
                    device=0 if device == 'cuda' else -1)
    print('[v5] DA V2 Small 로드 (inverse depth)')
    return {'kind': 'da_v2', 'pipeline': pipe, 'device': device}


def predict_depth(depth_obj, img_np):
    """Returns (depth_map, focal_px or None, is_metric)."""
    if depth_obj is None:
        return None, None, False
    H, W = img_np.shape[:2]

    if depth_obj['kind'] == 'depth_pro':
        proc = depth_obj['processor']
        model = depth_obj['model']
        device = depth_obj['device']
        pil = Image.fromarray(img_np)
        inputs = proc(images=pil, return_tensors='pt')
        if device == 'cuda':
            inputs = {k: (v.to(device).half() if v.dtype == torch.float32 else v.to(device))
                      for k, v in inputs.items()}
        else:
            inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            out = model(**inputs)
        post = proc.post_process_depth_estimation(out, target_sizes=[(H, W)])
        depth = post[0]['predicted_depth'].cpu().numpy().astype(np.float32)
        depth = np.nan_to_num(depth, nan=100.0, posinf=100.0, neginf=0.5)
        depth = np.clip(depth, 0.5, 100.0)
        focal_px = float(post[0].get('focal_length', 0.0))
        if focal_px <= 0: focal_px = W * 0.8
        return depth, focal_px, True

    elif depth_obj['kind'] == 'da_v2':
        pil = Image.fromarray(img_np)
        result = depth_obj['pipeline'](pil)
        depth = np.array(result['depth']).astype(np.float32)
        if depth.shape != (H, W):
            depth = cv2.resize(depth, (W, H), interpolation=cv2.INTER_LINEAR)
        return depth, None, False

    return None, None, False

---

# 9. Main Pipeline — `assess_v5()` + `draw_hud_v5()`

전체 파이프라인 (lane → vp → ego → distance → zone → 4 warning → max bin) 과 HUD 시각화.

In [ ]:
def get_ego_mask(mask):
    """My bike (5) + Rider (6) → close 5x5 → dilate 5x5 → CC closest to (W/2, H).
    Returns (H, W) uint8."""
    mh, mw = mask.shape
    bike_base = ((mask == 5) | (mask == 6)).astype(np.uint8)
    if bike_base.sum() < 200:
        return np.zeros_like(bike_base)
    k5 = np.ones((5, 5), np.uint8)
    closed = cv2.morphologyEx(bike_base, cv2.MORPH_CLOSE, k5)
    dilated = cv2.dilate(closed, np.ones((5, 5), np.uint8), iterations=1)
    n_cc, labels = cv2.connectedComponents(dilated, connectivity=8)
    if n_cc <= 1:
        return dilated
    anchor = np.array([mw / 2, mh])
    best_cc, best_dist = 1, float('inf')
    for cc_id in range(1, n_cc):
        ys_cc, xs_cc = np.where(labels == cc_id)
        if len(xs_cc) < 200: continue
        centroid = np.array([xs_cc.mean(), ys_cc.mean()])
        dist = np.linalg.norm(centroid - anchor)
        if dist < best_dist:
            best_dist = dist; best_cc = cc_id
    return (labels == best_cc).astype(np.uint8)


def assess_v5(img_np, mask, depth_map, img_name='', focal_px=None,
              depth_is_metric=True):
    """v5 메인 파이프라인.

    img_np:        (H, W, 3) RGB uint8
    mask:          (H, W) uint8, 7-class
    depth_map:     (H, W) float32 또는 None
    focal_px:      Depth Pro 의 focal (metric 시), 또는 None → 차선폭 역산 시도
    depth_is_metric: True (Depth Pro), False (DA V2)
    """
    H, W = mask.shape

    # 1. Lane → VP → Ego lane
    lines = detect_lane_lines(mask)
    vp_x, vp_y, vp_conf = vp_from_lanes(lines, W, H)
    if vp_conf < 0.3:
        vp_x, vp_y = W // 2, int(H * 0.4)
    ego = identify_ego_lane(lines, W, H)

    # 2. Distance estimator init
    est = DistanceEstimator(depth_is_metric=depth_is_metric)
    if focal_px is not None and focal_px > 0:
        est.f_px = float(focal_px)
        f_ok = True
    else:
        f_ok = est.calibrate(lines, vp_y, W, H)

    # 3. Object extraction + ego mask filter
    all_objs = extract_objects(mask)
    raw_moveables = [o for o in all_objs if o['class_id'] == 4]

    ego_mask = get_ego_mask(mask)

    def is_ego_fragment(bbox):
        if ego_mask.sum() == 0: return False
        x0, y0, x1, y1 = bbox
        patch = ego_mask[y0:y1+1, x0:x1+1]
        overlap = int(patch.sum())
        bbox_px = max((x1 - x0) * (y1 - y0), 1)
        thr = 0.55 if (x1 - x0) * (y1 - y0) >= 3500 else 0.40
        return overlap / bbox_px > thr

    moveables = [o for o in raw_moveables if not is_ego_fragment(o['bbox'])]
    n_filtered = len(raw_moveables) - len(moveables)

    if depth_map is not None and len(moveables) >= 2:
        est.calibrate_depth_alpha(depth_map, moveables, vp_y)

    # 4. Zone corridor
    corridor = None
    if ego['left'] is not None and ego['right'] is not None and vp_conf >= 0.3:
        y_bot = H - 1
        Lx_bot = line_x_at(ego['left'], y_bot)
        Rx_bot = line_x_at(ego['right'], y_bot)
        corridor = build_corridor(vp_x, vp_y, Lx_bot, Rx_bot, H, W,
                                   vp_conf=vp_conf)

    # 5. Per-object distance + zone
    objs_with_dist = []
    for o in moveables:
        info = est.estimate(o['bbox'], o['class_id'], depth_map, vp_y)
        cx_rel = o['cx'] / W
        zone_sev, zone_info = classify_zone(o['bbox'], corridor)
        objs_with_dist.append({
            'bbox': o['bbox'], 'class_id': o['class_id'],
            'cx': o['cx'], 'cy': o['cy'], 'cx_rel': cx_rel,
            'dist_info': info,
            'zone_sev': zone_sev, 'zone_info': zone_info,
        })

    # 6. 방향별 분류 + violation
    front_objs = [o for o in objs_with_dist if 0.35 <= o['cx_rel'] <= 0.65]
    left_objs = [o for o in objs_with_dist if o['cx_rel'] < 0.35]
    right_objs = [o for o in objs_with_dist if o['cx_rel'] > 0.65]

    strict_violators = []
    x1_violators = []
    if ego['left'] and ego['right']:
        for o in objs_with_dist:
            if check_lane_viol_strict(o['bbox'], ego, H):
                strict_violators.append(o)
    for o in objs_with_dist:
        if check_lane_viol_x1(o['bbox'], W):
            x1_violators.append(o)

    # 7. ego_x = ego_mask centroid (실제 my bike 위치)
    if ego_mask.sum() > 200:
        ys_em, xs_em = np.where(ego_mask > 0)
        ego_cx = float(xs_em.mean())
    else:
        ego_cx = None

    # 8. 4 warnings
    fcw = detect_fcw(front_objs, W, H)
    ldw = detect_ldw(ego, W, H, vp_y, ego_x=ego_cx)
    bsw = detect_bsw(left_objs, right_objs, strict_violators, x1_violators)
    head = detect_head(front_objs, strict_violators, x1_violators, W, H)

    # 9. Zone clamp patches
    zone_adjustments = []
    if corridor is not None:
        # Patch 1: FCW L3 by area_ratio override → zone 안일 때만 유지 (cut-in 예외)
        cut_in_scenario = (bsw.level >= 2 or head.level >= 2)
        if (fcw.level == 3 and fcw.extra.get('area_ratio', 0) > 0.25
                and fcw.bbox is not None and not cut_in_scenario):
            matching = next(
                (o for o in objs_with_dist if tuple(o['bbox']) == tuple(fcw.bbox)),
                None)
            if matching is not None and matching['zone_sev'] < 2:
                fcw = WarningResult(
                    'FCW', 2, fcw.reason + ' [zone clamp→L2]',
                    bbox=fcw.bbox,
                    extra={**fcw.extra, 'zone_clamped': True,
                           'zone_sev': matching['zone_sev']})
                zone_adjustments.append('FCW L3→L2 (zone clamp, no cut-in)')

        # Patch 2: BSW sidelobe boost
        if bsw.level == 0:
            for o in objs_with_dist:
                sl_hit = o['zone_info'].get('sidelobe_hit')
                if sl_hit and o['zone_info'].get('sidelobe_ratio', 0) > 0.45:
                    d = o['dist_info'].get('d_fused_m')
                    d_str = f'{d:.1f}m' if d else '거리불명'
                    bsw = WarningResult(
                        'BSW', 2, f'{sl_hit} sidelobe {d_str} [zone boost]',
                        bbox=o['bbox'],
                        extra={'sidelobe': True, 'side': sl_hit,
                               'zone_boost': True, 'distance_m': d})
                    zone_adjustments.append(f'BSW L0→L2 ({sl_hit} sidelobe)')
                    break

    # 10. Distance summary
    def closest_d(lst):
        ds = [o['dist_info']['d_fused_m'] for o in lst
              if o['dist_info']['d_fused_m'] is not None]
        return min(ds) if ds else None

    state = V5State(
        fcw=fcw, ldw=ldw, bsw=bsw, head=head,
        d_front_m=closest_d(front_objs),
        d_left_m=closest_d(left_objs),
        d_right_m=closest_d(right_objs),
        f_calibrated=f_ok,
        f_px=round(est.f_px, 1) if est.f_px else None,
    )

    # 2m 이내는 ego 오분류 가정 — warning 억제 + 거리값 clamp
    suppress_short_range_warnings(state)
    # HEAD 사후 검증: FCW 동시 활성 & 근거리 일치하지 않으면 억제 (극보수화)
    if state.head.level > 0:
        fcw_ok = state.fcw.level > 0 and (
            state.fcw.extra and state.fcw.extra.get('distance_m', 999) <= 4.5)
        if not fcw_ok:
            state.head.level = 0
            state.head.reason = '[억제] FCW 동시 활성 조건 미충족 (역주행 확신 부족)'
            state.head.bbox = None
    # [removed] lane violator → BSW 승격 — FP 많아서 제거. 필요 시 다시 활성화.
    # promote_lane_violators_to_bsw(state.bsw, len(strict_violators), len(x1_violators))

    return {
        'image': img_name,
        'bin': state.bin,
        'max_severity': state.max_severity,
        'warnings': {
            'FCW': fcw.to_dict(), 'LDW': ldw.to_dict(),
            'BSW': bsw.to_dict(), 'HEAD': head.to_dict(),
        },
        'active_warnings': [w.name for w in state.active_warnings],
        'top_warning': state.top_warning.name if state.top_warning else None,
        'top_reason': state.top_warning.reason if state.top_warning else '',
        'd_front_m': round(state.d_front_m, 2) if state.d_front_m else None,
        'd_left_m': round(state.d_left_m, 2) if state.d_left_m else None,
        'd_right_m': round(state.d_right_m, 2) if state.d_right_m else None,
        'f_calibrated': state.f_calibrated,
        'f_px': state.f_px,
        'vp': {'x': int(vp_x), 'y': int(vp_y), 'conf': round(float(vp_conf), 3)},
        'n_lines': len(lines),
        'n_moveable': len(moveables),
        'n_strict_viol': len(strict_violators),
        'n_x1_viol': len(x1_violators),
        'n_ego_fragments_filtered': n_filtered,
        'corridor_active': corridor is not None,
        'zone_adjustments': zone_adjustments,
        'per_obj_zones': [
            {'bbox': list(o['bbox']), 'zone_sev': o['zone_sev'],
             'anchor': o['zone_info']['anchor_zone'],
             'sidelobe': o['zone_info'].get('sidelobe_hit')}
            for o in objs_with_dist
        ],
        '_state': state,
        '_ego': ego,
        '_lines': lines,
        '_objs': objs_with_dist,
        '_corridor': corridor,
    }


def draw_hud_v5(ax, img, result):
    """HUD 오버레이를 matplotlib axis 에 그리기."""
    import matplotlib.patches as mpatches
    H, W = img.shape[:2]
    ax.imshow(img)
    state: V5State = result['_state']
    ego = result.get('_ego', {})
    bin_ = state.bin
    bin_color = BIN_COLORS[bin_]

    # Zone overlay
    corridor = result.get('_corridor')
    if corridor is not None:
        draw_zones_on_ax(ax, corridor)

    # Ego lane band — LDW level 따라 색상
    if ego.get('left') and ego.get('right'):
        y_top = max(result['vp']['y'], 0)
        ys = np.linspace(y_top, H, 30)
        xs_l = [line_x_at(ego['left'], y) for y in ys]
        xs_r = [line_x_at(ego['right'], y) for y in ys]
        ldw_lv = state.ldw.level
        if ldw_lv == 2: lane_c, lane_alpha = 'red', 0.20
        elif ldw_lv == 1: lane_c, lane_alpha = 'yellow', 0.15
        else: lane_c, lane_alpha = 'lime', 0.08
        ax.fill_betweenx(ys, xs_l, xs_r, color=lane_c, alpha=lane_alpha)
        ax.plot(xs_l, ys, '-', color=lane_c, lw=1.0, alpha=0.5)
        ax.plot(xs_r, ys, '-', color=lane_c, lw=1.0, alpha=0.5)
        if ldw_lv >= 1:
            ego_cx_draw = state.ldw.extra.get('ego_x', W/2)
            ax.plot([ego_cx_draw, ego_cx_draw], [H*0.80, H*0.98],
                    '-', color=lane_c, lw=3)
            ax.text(ego_cx_draw, H*0.79, state.ldw.reason, color=lane_c,
                    fontsize=7, weight='bold', ha='center', va='bottom',
                    bbox=dict(facecolor='black', alpha=0.7, pad=1))

    # 활성 경고 bbox (HEAD/FCW/BSW 우선순위)
    drawn = set()
    for w in [state.head, state.fcw, state.bsw]:
        if w.level == 0 or w.bbox is None: continue
        bbox_key = tuple(w.bbox)
        if bbox_key in drawn: continue
        drawn.add(bbox_key)
        x0, y0, x1, y1 = w.bbox
        c = WARNING_COLORS[w.name]
        lw = 1.5 + w.level * 0.8
        ax.add_patch(mpatches.Rectangle(
            (x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor=c, linewidth=lw))
        ax.text(x0, max(y0 - 8, 12),
                f"[{w.name}-L{w.level}] {w.reason}",
                color='white', fontsize=7, weight='bold',
                bbox=dict(facecolor=c, alpha=0.9, pad=2, edgecolor='none'))

    # 비경고 bbox 회색
    for o in result.get('_objs', []):
        if tuple(o['bbox']) in drawn: continue
        x0, y0, x1, y1 = o['bbox']
        ax.add_patch(mpatches.Rectangle(
            (x0, y0), x1 - x0, y1 - y0, fill=False, edgecolor='lightgray',
            linewidth=0.8, alpha=0.6))

    # 상단 bin bar
    ax.add_patch(mpatches.Rectangle((0, 0), W, H * 0.09,
                                     facecolor='black', alpha=0.85))
    ax.text(W * 0.02, H * 0.045, f"[{bin_}]", color=bin_color,
            fontsize=14, weight='bold', va='center')
    x_offset = W * 0.12
    for w in state.active_warnings:
        c = WARNING_COLORS[w.name]
        badge = f"{w.name}-{w.level}"
        ax.text(x_offset, H * 0.045, badge, color=c, fontsize=9, weight='bold',
                va='center',
                bbox=dict(facecolor='black', alpha=0.7, pad=2, edgecolor=c, lw=1))
        x_offset += len(badge) * W * 0.009 + W * 0.02

    # 하단 거리 요약
    ax.add_patch(mpatches.Rectangle((0, H * 0.91), W, H * 0.09,
                                     facecolor='black', alpha=0.85))
    parts = []
    for side, d in [('정면', state.d_front_m),
                    ('좌', state.d_left_m), ('우', state.d_right_m)]:
        parts.append(f'{side}:{d:.1f}m' if d is not None else f'{side}:-')
    f_str = (f"f={state.f_px:.0f}" + ("✓" if state.f_calibrated else "~")
             if state.f_px else "")
    ax.text(W * 0.02, H * 0.955, '   '.join(parts), color='#FFDE59',
            fontsize=9, va='center', weight='bold')
    if f_str:
        ax.text(W * 0.98, H * 0.955, f_str, color='#7FFFD4', fontsize=8,
                va='center', ha='right')

    ax.set_xlim(0, W); ax.set_ylim(H, 0)
    ax.axis('off')

---

# 10. Public API — `RiskAssessorV5`

팀원이 실제로 사용하는 인터페이스. 모델 로드는 한 번, 평가는 반복 호출.

**핵심 API** (기본 사용 순서):

- `RiskAssessorV5(seg_checkpoints=None, use_depth='depth_pro', device='cuda')`
  인스턴스 생성. `seg_checkpoints=None` 이 기본 — 외부 mask 주입 모드.
- `assess_with_mask(image, mask, depth=None)` — **기본 사용**: 팀원 모델 mask 주입
- `assess(image)` — 내장 U-Net 으로 seg 부터 end-to-end (우리 팀 체크포인트 있을 때)
- `draw_hud(image, result, save_path=None)` — HUD 시각화 (matplotlib fig 반환 또는 저장)
- `result_to_json(result)` — 내부 객체 키 제외한 JSON-직렬화 dict 반환


In [ ]:
class RiskAssessorV5:
    """v5 위험도 평가 시스템 — 단일 클래스 인터페이스.

    Pipeline:
        Image + Mask → Lane/VP → Depth(Pro/V2/None) → Distance(3-est)
                     → Zone → 4 Warning(FCW/LDW/BSW/HEAD) → bin = max(severities)

    기본 사용 시나리오: **팀원이 자기 모델로 segment 한 7-class mask 를 주입**.
    assess_with_mask(image, mask) 로 호출하면 파이프라인이 그 위에서 동작.

    Args:
        seg_checkpoints:  기본 None — 외부 mask 주입 (assess_with_mask) 모드.
                          우리 팀 내장 U-Net 쓰려면 단일 경로(str) 또는 경로 리스트(5-fold).
        use_depth:        'depth_pro' (기본, metric, 권장) | 'da_v2' (inverse) | None.
                          None 이어도 image+mask 로 동작 (거리 = ground+size only).
        device:           'cuda' | 'cpu' (auto fallback 도 함).
    """

    def __init__(self, seg_checkpoints=None, use_depth='depth_pro', device='cuda'):
        self.device = device if torch.cuda.is_available() else 'cpu'
        if device == 'cuda' and self.device != 'cuda':
            print('[v5] CUDA 미감지 → CPU 사용 (느릴 수 있음)')

        # Seg 로드
        if seg_checkpoints is None:
            self.seg_models = None
            print('[v5] seg_checkpoints=None (기본) — assess_with_mask() 로 외부 mask 주입하세요')
        elif isinstance(seg_checkpoints, str):
            self.seg_models = [load_seg_model(seg_checkpoints, self.device)]
        else:
            self.seg_models = load_seg_ensemble(list(seg_checkpoints), self.device)
            if len(self.seg_models) == 0:
                raise RuntimeError('Ensemble 체크포인트 모두 누락')

        # Depth 로드
        self.depth_mode = use_depth
        if use_depth == 'depth_pro':
            self.depth_obj = load_depth_pro(self.device)
        elif use_depth == 'da_v2':
            self.depth_obj = load_depth_v2(self.device)
        elif use_depth is None:
            self.depth_obj = None
            print('[v5] depth 비활성 — 거리는 ground+size estimator 만 사용')
        else:
            raise ValueError(
                f"use_depth='{use_depth}' 미지원. "
                "'depth_pro' | 'da_v2' | None 중 선택")

        n_seg = len(self.seg_models) if self.seg_models else 0
        mode = 'external mask' if n_seg == 0 else f'internal U-Net ({n_seg} models)'
        print(f'[v5] 초기화 완료 — seg mode: {mode}, depth={use_depth}, '
              f'device={self.device}')

    def assess(self, image):
        """내장 U-Net 으로 seg 부터 end-to-end.
        우리 팀 체크포인트를 로드한 경우에만 사용 가능.
        외부 mask 주입이 기본이므로 보통은 assess_with_mask() 를 쓰세요.

        image: 파일 경로(str/Path) 또는 RGB numpy array (H,W,3)."""
        if self.seg_models is None:
            raise RuntimeError(
                'seg_checkpoints=None 이므로 assess() 불가. '
                'assess_with_mask(image, mask) 를 사용하거나 '
                'seg_checkpoints 로 체크포인트 경로를 지정하세요.')
        img_np = self._load_image(image)
        mask = predict_seg_ensemble(self.seg_models, img_np, self.device)
        if self.depth_obj is not None:
            depth, focal_px, is_metric = predict_depth(self.depth_obj, img_np)
        else:
            depth, focal_px, is_metric = None, None, False
        img_name = str(image) if isinstance(image, (str, Path)) else ''
        return assess_v5(img_np, mask, depth, img_name=img_name,
                         focal_px=focal_px, depth_is_metric=is_metric)

    def assess_with_mask(self, image, mask, depth=None, focal_px=None,
                         depth_is_metric=False):
        """**기본 사용 API** — 외부 mask 주입.

        팀원이 다른 아키텍처로 segment 한 7-class mask 를 직접 넣을 때 사용.
        mask 클래스 규약: 0=BG, 1=Undrivable, 2=Road, 3=LaneMark,
                         4=Moveable, 5=MyBike, 6=Rider.

        Args:
            image:  파일 경로(str/Path) 또는 RGB numpy array (H,W,3).
            mask:   (H,W) uint8 — image 와 동일 H,W.
            depth:  None 이면 내부 depth 모델로 자동 계산. 본인 depth 있으면 주입 가능.
            focal_px:       depth 같이 주입 시 focal (px). 없으면 내부 추정.
            depth_is_metric: depth 가 metric(m) 이면 True, inverse(DA V2-like) 면 False.
        """
        img_np = self._load_image(image)
        if mask.shape[:2] != img_np.shape[:2]:
            raise ValueError(
                f'mask shape {mask.shape} != image shape {img_np.shape[:2]}')
        # Depth 안 줬으면 내부 모델로 계산
        if depth is None and self.depth_obj is not None:
            depth, focal_px_pred, depth_is_metric = predict_depth(
                self.depth_obj, img_np)
            if focal_px is None: focal_px = focal_px_pred
        return assess_v5(img_np, mask.astype(np.uint8), depth,
                         focal_px=focal_px, depth_is_metric=depth_is_metric)

    def draw_hud(self, image, result, save_path=None, dpi=110):
        """HUD 시각화. save_path 주면 파일 저장, 아니면 fig 반환."""
        import matplotlib.pyplot as plt
        img_np = self._load_image(image)
        H, W = img_np.shape[:2]
        fig, ax = plt.subplots(figsize=(W/100, H/100), dpi=dpi)
        fig.subplots_adjust(left=0, right=1, top=1, bottom=0)
        draw_hud_v5(ax, img_np, result)
        if save_path:
            fig.savefig(save_path, dpi=dpi, bbox_inches='tight', pad_inches=0)
            plt.close(fig)
            print(f'[v5] HUD 저장: {save_path}')
            return save_path
        return fig

    def result_to_json(self, result):
        """JSON 직렬화 가능한 dict 만 추출 (내부 객체 키 제거)."""
        return {k: v for k, v in result.items() if not k.startswith('_')}

    @staticmethod
    def _load_image(image):
        if isinstance(image, (str, Path)):
            return np.array(Image.open(image).convert('RGB'))
        elif isinstance(image, np.ndarray):
            return image
        elif hasattr(image, 'mode'):  # PIL
            return np.array(image.convert('RGB'))
        raise TypeError(f'Unsupported image type: {type(image)}')


---

# 11. 사용 예시 — 기본: 외부 mask 주입

**주 시나리오**: 팀원이 다른 아키텍처로 학습한 모델의 7-class mask 를 주입하여 위험도 평가.

본인 환경에 맞춰 아래 `CONFIG` 부분만 수정 후 실행하세요.

**예상 첫 실행 시간** (Depth Pro + 외부 mask 모드):
- Apple Depth Pro 다운로드 (1GB): 1~5분 (네트워크 따라, 처음 한 번만)
- 인스턴스 생성: 5~10초 (depth 로드만)
- 이미지 평가: ~0.3초/장 (RTX 4070), ~1초/장 (T4)

**우리 팀 내장 U-Net 을 쓰려는 경우** → 섹션 12.1 참고.


In [ ]:
# ============================================================
# CONFIG — 본인 환경에 맞춰 수정
# ============================================================

# (1) 이미지 & 외부 mask
#     - IMAGE_PATH : 평가할 이미지
#     - MASK       : 본인 모델의 segmentation 출력 (H, W) uint8
#                    클래스 규약: 0=BG, 1=Undrivable, 2=Road, 3=LaneMark,
#                                4=Moveable, 5=MyBike, 6=Rider
IMAGE_PATH = 'frame.png'

# 옵션 A) mask 를 .npy / .png 파일로 저장해 둔 경우
# MASK = np.load('my_mask.npy').astype(np.uint8)
# MASK = np.array(Image.open('my_mask.png'))

# 옵션 B) 본인 모델을 여기서 직접 호출
# image = np.array(Image.open(IMAGE_PATH).convert('RGB'))
# MASK = my_segmentation_model(image).astype(np.uint8)

# (아래는 placeholder — 실제로는 위 옵션 A 또는 B 로 교체)
MASK = np.zeros((720, 1280), dtype=np.uint8)

# (2) Depth 모드
#     - 'depth_pro' : Apple Depth Pro (metric m, 정확, 1GB, 권장)
#     - 'da_v2'     : Depth Anything V2 Small (inverse, ~100MB)
#     - None        : Depth 사용 안 함 (거리 = ground+size estimator only)
DEPTH_MODE = 'depth_pro'

# (3) Device
DEVICE = 'cuda'  # or 'cpu'

# ============================================================
# 인스턴스 생성 (한 번만) — 외부 mask 모드
# ============================================================
assessor = RiskAssessorV5(
    seg_checkpoints=None,   # ← 외부 mask 주입 모드 (기본)
    use_depth=DEPTH_MODE,
    device=DEVICE,
)

# ============================================================
# 단일 이미지 평가 — assess_with_mask
# ============================================================
result = assessor.assess_with_mask(IMAGE_PATH, MASK)

print(f"\n=== Risk Assessment ===")
print(f"Bin           : {result['bin']}  (max_severity={result['max_severity']})")
print(f"Active        : {result['active_warnings']}")
print(f"Top warning   : {result['top_warning']} - {result['top_reason']}")
print(f"Distances     : 정면={result['d_front_m']}m  "
      f"좌={result['d_left_m']}m  우={result['d_right_m']}m")
print(f"Focal         : {result['f_px']}px  (calibrated={result['f_calibrated']})")
print(f"VP            : ({result['vp']['x']}, {result['vp']['y']})  "
      f"conf={result['vp']['conf']}")
print(f"Moveables     : {result['n_moveable']} "
      f"(ego fragments filtered: {result['n_ego_fragments_filtered']})")
print(f"Lane lines    : {result['n_lines']}")
print(f"Violators     : strict={result['n_strict_viol']}  x1={result['n_x1_viol']}")
print(f"Corridor      : {'ON' if result['corridor_active'] else 'OFF'}")
if result['zone_adjustments']:
    print(f"Zone patches  : {result['zone_adjustments']}")

# 각 warning 상세
print(f"\n--- Per-Warning Detail ---")
for name, w in result['warnings'].items():
    print(f"  {name}: L{w['level']}  {w['reason']}")

# ============================================================
# HUD 시각화
# ============================================================
import matplotlib.pyplot as plt
fig = assessor.draw_hud(IMAGE_PATH, result)
plt.show()

# 파일로 저장하려면:
# assessor.draw_hud(IMAGE_PATH, result, save_path='hud_output.png')

# ============================================================
# JSON 저장 (내부 객체 제외)
# ============================================================
# import json
# with open('result.json', 'w', encoding='utf-8') as f:
#     json.dump(assessor.result_to_json(result), f,
#               indent=1, ensure_ascii=False)


---

# 12. 추가 사용 패턴

## 12.1 우리 팀 내장 U-Net 사용 (옵션)

우리 팀이 학습한 Lane Focus U-Net R34 5-fold 체크포인트를 사용하려면 `seg_checkpoints` 를 지정하고 `assess()` 를 쓰면 됩니다. Seg + Depth 를 한 번에 end-to-end 로 돌립니다.

**체크포인트 배포**: `lane_focus_fold0_best.pth ~ lane_focus_fold4_best.pth` (각 ~93MB). Hugging Face Hub 에서 받는다:

```python
from huggingface_hub import hf_hub_download

OUR_CKPT_PATHS = [
    hf_hub_download('minoak/motorcycle-night-lane-focus', f'lane_focus_fold{i}_best.pth')
    for i in range(5)
]
```


In [ ]:
# 우리 팀 내장 U-Net 5-fold ensemble
OUR_CKPT_PATHS = [
    'lane_focus_fold0_best.pth',
    'lane_focus_fold1_best.pth',
    'lane_focus_fold2_best.pth',
    'lane_focus_fold3_best.pth',
    'lane_focus_fold4_best.pth',
]
# 또는 단일 fold: OUR_CKPT_PATHS = 'lane_focus_fold0_best.pth'

assessor_internal = RiskAssessorV5(
    seg_checkpoints=OUR_CKPT_PATHS,
    use_depth='depth_pro',
)

# assess() 사용 — seg 부터 end-to-end
result = assessor_internal.assess(IMAGE_PATH)
print(result['bin'], result['active_warnings'])


## 12.2 폴더 일괄 평가

한 폴더의 이미지를 모두 평가하고 결과를 JSON 으로 저장:

In [ ]:
from pathlib import Path
import json

INPUT_DIR = Path('test_images')
MASK_DIR = Path('test_masks')          # 팀원 모델로 미리 저장해 둔 mask .npy 들
OUTPUT_JSON = 'batch_results.json'

results = {}
for img_path in sorted(INPUT_DIR.glob('*.png')):
    mask_path = MASK_DIR / (img_path.stem + '.npy')
    if not mask_path.exists():
        print(f'[skip] mask 없음: {mask_path.name}')
        continue
    mask = np.load(mask_path).astype(np.uint8)
    r = assessor.assess_with_mask(str(img_path), mask)
    results[img_path.name] = assessor.result_to_json(r)
    print(f"{img_path.name}: {r['bin']}  {r['active_warnings']}")

with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
    json.dump(results, f, indent=1, ensure_ascii=False)
print(f'\n결과 저장: {OUTPUT_JSON}')

# ---
# 내장 U-Net 쓰는 경우에는 mask 준비 불필요:
# for img_path in sorted(INPUT_DIR.glob('*.png')):
#     r = assessor_internal.assess(str(img_path))
#     ...


## 12.3 임계값 커스터마이징

위험도 임계값을 바꾸려면 노트북 상단 `Part 2` 의 상수를 수정 후 모든 셀 재실행:

```python
# 예: FCW 더 보수적으로
FCW_DIST_CRITICAL = 2.0   # 3.0 → 2.0
FCW_DIST_WARNING = 4.0    # 5.0 → 4.0
```

## 12.4 결과 dict 구조

```python
result = {
    'bin': str,                         # '안전' | '주의' | '위험' | '치명'
    'max_severity': int,                # 0~3
    'active_warnings': list[str],       # ['FCW', 'BSW', ...]
    'top_warning': str,                 # 가장 심각한 모듈 이름
    'top_reason': str,                  # 한글 설명
    'warnings': {                       # 모든 모듈 상세
        'FCW': {'name', 'level', 'reason', 'bbox', 'distance_m', ...},
        'LDW': {...}, 'BSW': {...}, 'HEAD': {...},
    },
    'd_front_m': float | None,
    'd_left_m':  float | None,
    'd_right_m': float | None,
    'f_px': float, 'f_calibrated': bool,
    'vp': {'x', 'y', 'conf'},
    'n_lines', 'n_moveable', 'n_strict_viol', 'n_x1_viol',
    'corridor_active': bool,
    'zone_adjustments': list[str],
    'per_obj_zones': list[dict],
    # 내부 (JSON 저장 시 제외):
    '_state', '_ego', '_lines', '_objs', '_corridor',
}
```

---

# 13. 트러블슈팅

| 증상 | 원인 / 해결 |
|---|---|
| `apple/DepthPro-hf` 모델 로드 실패 | `transformers >= 4.40` 필요. `pip install -U transformers` |
| `weights_only` 관련 에러 | PyTorch 버전 차이. 노트북의 `_safe_torch_load` 가 자동 처리하지만 직접 `torch.load` 호출 시 주의 |
| CUDA OOM | DEPTH_MODE='da_v2' 또는 None 으로 변경, 또는 단일 fold 사용 |
| HUD 한글 박스로 보임 | 폰트 부재. Linux/Mac 에서 `Noto Sans CJK KR` 또는 `NanumGothic` 설치 |
| `albumentations` import 에러 | `pip install albumentations` (Python 3.12 환경에서 가끔 자동 설치 안 됨) |
| 첫 실행이 매우 느림 | Depth Pro 1GB 다운로드 중. 한 번만 일어남 (`~/.cache/huggingface/hub/`) |
| 결과의 `_state` 등으로 JSON 저장 안 됨 | `assessor.result_to_json(result)` 호출 |
| ego 차량 자체가 BSW 로 잡힘 | 정상 동작 — `n_ego_fragments_filtered` 로 필터링된 개수 확인 |